## Lib integration

In [1]:
import sys
# sys.path.append(r"E:\Dai hoc\2526I\dacn\flow-matching")
sys.path.append(r"E:\Dai hoc\2526I\dacn\flow-matching\demo-code\2d")
import h5py
from collections import defaultdict, Counter
import numpy as np
from rich import print

In [2]:
file_path = r"E:\Dai hoc\2526I\dacn\flow-matching\data\traintest_hcd.hdf5"

## Open data

In [3]:
with h5py.File(file_path, "r") as f:
    print("Keys:", list(f.keys()))

    seqs = f["sequence_integer"][:10024]
    charges_oh = f["precursor_charge_onehot"][:10024]
    intensities = f["intensities_raw"][:10024]  

Keys:
[
    'collision_energy',
    'collision_energy_aligned',
    'collision_energy_aligned_normed',
    'intensities_raw',
    'masses_pred',
    'masses_raw',
    'method',
    'precursor_charge_onehot',
    'rawfile',
    'reverse',
    'scan_number',
    'score',
    'sequence_integer',
    'sequence_onehot'
]

In [4]:
charges = np.argmax(charges_oh, axis=1) + 1
del charges_oh

### Utils

### Peak data

In [5]:
# np.argmax(charges_oh, axis=1) + 1

In [6]:
len(intensities[0])

174

In [7]:
# max_index = 0
# for seq in seqs:
#     for token in seq:
#         if token > max_index:
#             max_index = token
# print("Max token index:", max_index)

## Explore data

## FLow matching training

In [8]:
import torch
torch.set_default_device("cuda")
from torch import nn
# from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
import imageio
from sklearn.datasets import make_moons
import math

from coupling import mini_batch_coupling, greedy_coupling, sinkhorm_coupling
from draw import DrawFlow, DrawSample
from gen_path import get_xt
from flow_model import HCDFlowResMLP, HCDFlow

### Training configuration

In [9]:
epoch = 1e6
batch_size = 1024
model = HCDFlow(noise_dim=174, pep_dim=256, time_dim=128, charge_dim=128)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, eps=1e-10)

#### Tranable params

In [10]:
sum(p.numel() for p in model.parameters() if p.requires_grad)

5615790

In [11]:
# from tqdm.auto import tqdm
# # pbar = tqdm(range(int(epoch)), desc="Training")
# loss_history = []
# pbar = tqdm(range(int(epoch)), desc="Training")
# for ep in pbar:
#     model.train()
#     batch_indices = np.random.choice(len(intensities), batch_size, replace=False)
#     batch_intensities = torch.tensor(intensities[batch_indices], dtype=torch.float32)
#     batch_pep_seq = torch.tensor(seqs[batch_indices], dtype=torch.long)
#     batch_charge = torch.tensor(charges[batch_indices], dtype=torch.float32).unsqueeze(1)
    
#     noise = torch.randn_like(batch_intensities)
#     t = torch.rand(batch_size, 1)
#     x_t = get_xt(batch_intensities, noise, t)
#     # print(x_t.shape, t.shape, batch_pep_seq.shape, batch_charge.shape)
#     u_pred = model(x_t, t=t, pep_seq=batch_pep_seq, charge=batch_charge)
#     loss = nn.MSELoss()(u_pred, batch_intensities - noise)
#     optimizer.zero_grad()

#     loss.backward()

#     optimizer.step()

#     loss_history.append(loss.item())
#     if (ep + 1) % 100 == 0:
#        pbar.set_postfix({"Loss": f"{loss.item():.4f}", "Avg Loss 100": f"{(sum(loss_history)/len(loss_history)):.4f}"})

## Pearson Correlation Coefficent

In [24]:
# pcc cal
def pcc(Y_true: torch.Tensor, Y_pred: torch.Tensor):
    # Y_true and Y_pred dim are (B, D)
    Y_true_mean = Y_true.mean(dim=1, keepdim=True)
    Y_pred_mean = Y_pred.mean(dim=1, keepdim=True)
    Y_true_centered = Y_true - Y_true_mean
    Y_pred_centered = Y_pred - Y_pred_mean
    covariance = (Y_true_centered * Y_pred_centered).sum(dim=0)
    Y_true_std = Y_true_centered.pow(2).sum(dim=0).sqrt()
    Y_pred_std = Y_pred_centered.pow(2).sum(dim=0).sqrt()
    pcc = covariance / (Y_true_std * Y_pred_std + 1e-8)
    return pcc.mean().item()

In [13]:
test_seq = seqs[0]
test_charge = charges[0]
test_intensities = []
for seq, charge, intensity in zip(seqs, charges, intensities):
    
    if np.array_equal(seq, test_seq) and test_charge == charge:
        test_intensities.append(intensity)
len(test_intensities)

1

In [26]:
with torch.no_grad():
    model.eval()
    test_pep_seq = torch.tensor(test_seq, dtype=torch.long).unsqueeze(0)
    test_charge_tensor = torch.tensor(test_charge, dtype=torch.float32).unsqueeze(0).unsqueeze(1)
    test_intensities_tensor = torch.tensor(test_intensities, dtype=torch.float32)
    
    num_samples = len(test_intensities)
    noise = torch.randn(num_samples, *test_intensities_tensor[0].shape)
    
    test_pep_seq = test_pep_seq.repeat(num_samples, 1)
    test_charge_tensor = test_charge_tensor.repeat(num_samples, 1)
    pred_intentities = model.sample(noise,test_pep_seq, test_charge_tensor, step=10)
    # print(pred_intentities - test_intensities_tensor)
    
    pcc_value = pcc(test_intensities_tensor, pred_intentities)
    print(f"PCC: {pcc_value:.4f}")

PCC: 0.2069

In [19]:
test_intensities_tensor.shape

torch.Size([1, 174])